# Data Preparation

This notebook prepares the modeling table for hourly yellow taxi demand forecasting across NYC taxi zones.

The raw taxi files are trip-level parquet files, so the central transformation is:

`one row per taxi ride` -> `one row per pickup zone per hour`

External data is then joined at the same hourly or daily level:

- NOAA-style weather observations from selected NYC-area stations
- US federal holiday indicators
- Taxi zone metadata

## 1. Import libraries and define paths

The notebook uses pandas for data preparation and pyarrow through pandas for parquet I/O. The taxi files are processed month by month to avoid loading the full year into memory at once.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

PROJECT_DIR = Path.cwd()
RAW_TAXI_DIR = PROJECT_DIR / "data" / "data"
SELF_ADDED_DIR = PROJECT_DIR / "data" / "self_added_data"
ZONE_LOOKUP_PATH = PROJECT_DIR / "data" / "taxi_zone_lookup.csv"
WEATHER_PATH = SELF_ADDED_DIR / "2023_US_Weather_Hourly.csv"
HOLIDAY_PATH = SELF_ADDED_DIR / "US Federal Pay and Leave Holidays 2004 to 2100.csv"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

taxi_files = sorted(RAW_TAXI_DIR.glob("yellow_tripdata_2023-*.parquet"))
taxi_files

[WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/data/yellow_tripdata_2023-01.parquet'),
 WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/data/yellow_tripdata_2023-02.parquet'),
 WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/data/yellow_tripdata_2023-03.parquet'),
 WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/data/yellow_tripdata_2023-04.parquet'),
 WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/data/yellow_tripdata_2023-05.parquet'),
 WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/data/yellow_tripdata_2023-06.parquet'),
 WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/data/yellow_tripdata_2023-07.parquet'),
 WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-deman

## 2. Load and clean taxi zones

The taxi zone lookup gives readable borough and zone names. For modeling demand across NYC taxi zones, this notebook removes special IDs 264 and 265 because they represent unknown/outside-of-NYC locations rather than normal forecastable zones.

In [2]:
zones = pd.read_csv(ZONE_LOOKUP_PATH)

valid_zones = zones[zones["LocationID"].between(1, 263)].copy()

print(valid_zones.shape)
valid_zones.head()

(263, 4)


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


## 3. Clean taxi trips and aggregate to hourly demand

Cleaning rules used here:

- Keep pickups from 2023 only.
- Require dropoff time after pickup time.
- Remove impossible or extremely long trips above 4 hours.
- Keep pickup and dropoff zones 1-263.
- Remove negative distance and negative monetary values.

`passenger_count = 0` is not removed because it can mean unknown or unrecorded passenger count, while the record can still represent a real taxi trip.

The target variable for the demand model is `trip_count`: number of pickups in a zone during an hour.

In [3]:
taxi_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type",
]

monthly_aggregates = []
cleaning_summary = []

# Process one monthly parquet file at a time to avoid loading the full year into memory.
for path in taxi_files:
    month = path.stem.replace("yellow_tripdata_", "")
    df = pd.read_parquet(path, columns=taxi_cols)
    original_rows = len(df)

    df = df[
        (df["tpep_pickup_datetime"] >= "2023-01-01")
        & (df["tpep_pickup_datetime"] < "2024-01-01")
    ].copy()

    df = df[df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]].copy()
    df["trip_duration_min"] = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    df = df[(df["trip_duration_min"] > 0) & (df["trip_duration_min"] <= 240)]
    df = df[df["PULocationID"].between(1, 263)]
    df = df[df["DOLocationID"].between(1, 263)]
    df = df[df["trip_distance"] >= 0]
    df = df[df["fare_amount"] >= 0]
    df = df[df["total_amount"] >= 0]

    # Floor each pickup timestamp to the hour used as the forecasting time bucket.
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.floor("h")

    agg = (
        df.groupby(["pickup_hour", "PULocationID"], as_index=False)
        .agg(
            trip_count=("PULocationID", "size"),
            revenue=("total_amount", "sum"),
            avg_trip_distance=("trip_distance", "mean"),
            avg_fare_amount=("fare_amount", "mean"),
            avg_trip_duration_min=("trip_duration_min", "mean"),
            zero_passenger_count=("passenger_count", lambda s: (s == 0).sum()),
        )
    )

    monthly_aggregates.append(agg)
    cleaning_summary.append(
        {
            "month": month,
            "raw_rows": original_rows,
            "clean_rows": len(df),
            "removed_rows": original_rows - len(df),
            "removed_pct": round((original_rows - len(df)) / original_rows * 100, 2),
        }
    )

taxi_hourly_observed = pd.concat(monthly_aggregates, ignore_index=True)

taxi_hourly_observed = (
    taxi_hourly_observed.groupby(["pickup_hour", "PULocationID"], as_index=False)
    .agg(
        trip_count=("trip_count", "sum"),
        revenue=("revenue", "sum"),
        avg_trip_distance=("avg_trip_distance", "mean"),
        avg_fare_amount=("avg_fare_amount", "mean"),
        avg_trip_duration_min=("avg_trip_duration_min", "mean"),
        zero_passenger_count=("zero_passenger_count", "sum"),
    )
)

cleaning_summary = pd.DataFrame(cleaning_summary)
cleaning_summary

,month,raw_rows,clean_rows,removed_rows,removed_pct
0,2023-01,3066766,2981260,85506,2.79
1,2023-02,2913955,2833597,80358,2.76
2,2023-03,3403766,3317804,85962,2.53
3,2023-04,3288250,3200880,87370,2.66
4,2023-05,3513649,3420358,93291,2.66
5,2023-06,3307234,3216450,90784,2.75
6,2023-07,2907108,2821323,85785,2.95
7,2023-08,2824209,2741088,83121,2.94
8,2023-09,2846722,2765966,80756,2.84
9,2023-10,3522285,3426586,95699,2.72


## 4. Create a complete hourly zone grid

A missing zone-hour does not mean missing data; it means zero observed pickups. Creating a full grid prevents the model from learning only from busy zone-hours.

In [4]:
all_hours = pd.date_range("2023-01-01 00:00:00", "2023-12-31 23:00:00", freq="h")

grid = pd.MultiIndex.from_product(
    [all_hours, valid_zones["LocationID"]],
    names=["pickup_hour", "PULocationID"],
).to_frame(index=False)

taxi_hourly = grid.merge(
    taxi_hourly_observed,
    on=["pickup_hour", "PULocationID"],
    how="left",
)

zero_fill_cols = ["trip_count", "revenue", "zero_passenger_count"]
taxi_hourly[zero_fill_cols] = taxi_hourly[zero_fill_cols].fillna(0)

taxi_hourly = taxi_hourly.merge(
    valid_zones,
    left_on="PULocationID",
    right_on="LocationID",
    how="left",
)

taxi_hourly = taxi_hourly.drop(columns=["LocationID"])

print(taxi_hourly.shape)
taxi_hourly.head()

(2303880, 11)


,pickup_hour,PULocationID,trip_count,revenue,avg_trip_distance,avg_fare_amount,avg_trip_duration_min,zero_passenger_count,Borough,Zone,service_zone
0,2023-01-01,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR
1,2023-01-01,2,0.0,0.00,NaN,NaN,NaN,0.0,Queens,Jamaica Bay,Boro Zone
2,2023-01-01,3,0.0,0.00,NaN,NaN,NaN,0.0,Bronx,Allerton/Pelham Gardens,Boro Zone
3,2023-01-01,4,19.0,519.02,3.090526,18.677368,14.911404,0.0,Manhattan,Alphabet City,Yellow Zone
4,2023-01-01,5,0.0,0.00,NaN,NaN,NaN,0.0,Staten Island,Arden Heights,Boro Zone


## 5. Prepare NOAA-style weather observations

The weather file contains many US stations. For this project, I keep four NYC-area stations:

- `72505394728`: NY City Central Park, used for most taxi zones
- `72503014732`: LaGuardia Airport, used for taxi zone 138
- `74486094789`: JFK International Airport, used for taxi zone 132
- `72502014734`: Newark Liberty International Airport, used for taxi zone 1

The full NOAA-style `STATION` value is the concatenation of the USAF station ID and the WBAN ID. For example, Central Park is `725053` + `94728` = `72505394728`.

NOAA `DATE` timestamps are interpreted as UTC observations. They are converted to `America/New_York` local time before joining to taxi pickup hours because TLC taxi timestamps are local NYC time.

In [5]:
NOAA_STATIONS = {
    72505394728: {
        "weather_station_key": "central_park",
        "weather_station_name": "NY City Central Park, NY US",
        "usaf": "725053",
        "wban": "94728",
        "taxi_zone_note": "Default weather station for non-airport NYC taxi zones",
    },
    72503014732: {
        "weather_station_key": "laguardia",
        "weather_station_name": "LaGuardia Airport, NY US",
        "usaf": "725030",
        "wban": "14732",
        "taxi_zone_note": "Used for LaGuardia Airport taxi zone 138",
    },
    74486094789: {
        "weather_station_key": "jfk",
        "weather_station_name": "JFK International Airport, NY US",
        "usaf": "744860",
        "wban": "94789",
        "taxi_zone_note": "Used for JFK Airport taxi zone 132",
    },
    72502014734: {
        "weather_station_key": "newark",
        "weather_station_name": "Newark Liberty International Airport, NJ US",
        "usaf": "725020",
        "wban": "14734",
        "taxi_zone_note": "Used for Newark Airport taxi zone 1",
    },
}

station_metadata = pd.DataFrame.from_dict(NOAA_STATIONS, orient="index")
station_metadata.index.name = "STATION"
station_metadata = station_metadata.reset_index()
station_metadata

,STATION,weather_station_key,weather_station_name,usaf,wban,taxi_zone_note
0,72505394728,central_park,"NY City Central Park, NY US",725053,94728,Default weather station for non-airport NYC ta...
1,72503014732,laguardia,"LaGuardia Airport, NY US",725030,14732,Used for LaGuardia Airport taxi zone 138
2,74486094789,jfk,"JFK International Airport, NY US",744860,94789,Used for JFK Airport taxi zone 132
3,72502014734,newark,"Newark Liberty International Airport, NJ US",725020,14734,Used for Newark Airport taxi zone 1


### Decode NOAA compact fields

Several NOAA fields are stored as compact comma-separated strings. This notebook extracts modeling-friendly weather variables:

- `TMP`: air temperature in tenths of degrees Celsius
- `DEW`: dew point in tenths of degrees Celsius
- `SLP`: sea-level pressure in tenths of hectopascals
- `WND`: wind speed in tenths of meters per second
- `VIS`: visibility in meters
- `AA1`, `AA2`, `AA3`: liquid precipitation depth, converted to millimeters for 1-hour accumulation records

In [6]:
def parse_signed_tenths(value, missing_code):
    # NOAA stores values like +0057,5; the first part means 5.7 in tenths.
    if pd.isna(value):
        return np.nan

    raw = str(value).split(",")[0]

    try:
        number = int(raw)
    except ValueError:
        return np.nan

    if abs(number) == missing_code:
        return np.nan

    return number / 10


def parse_wind_speed_mps(value):
    # WND is comma-separated; the fourth field is wind speed in tenths of m/s.
    if pd.isna(value):
        return np.nan

    parts = str(value).split(",")
    if len(parts) < 4:
        return np.nan

    try:
        speed = int(parts[3])
    except ValueError:
        return np.nan

    if speed == 9999:
        return np.nan

    return speed / 10


def parse_visibility_m(value):
    if pd.isna(value):
        return np.nan

    raw = str(value).split(",")[0]

    try:
        distance = int(raw)
    except ValueError:
        return np.nan

    if distance == 999999:
        return np.nan

    return distance


def parse_one_hour_precip_mm(value):
    # AA fields include accumulation period and depth; I keep only 1-hour precipitation.
    if pd.isna(value):
        return np.nan

    parts = str(value).split(",")
    if len(parts) < 2:
        return np.nan

    try:
        period_hours = int(parts[0])
        depth = int(parts[1])
    except ValueError:
        return np.nan

    if period_hours != 1 or depth == 9999:
        return np.nan

    return depth / 10


def first_non_null_max(values):
    cleaned = [v for v in values if not pd.isna(v)]
    return max(cleaned) if cleaned else np.nan

### Filter, decode, and aggregate weather to station-hour level

Some stations report multiple observations inside the same hour. The notebook averages continuous measurements and takes the maximum 1-hour precipitation value to avoid double-counting special observations.

In [7]:
weather_cols = [
    "STATION",
    "DATE",
    "NAME",
    "TMP",
    "DEW",
    "WND",
    "VIS",
    "SLP",
    "AA1",
    "AA2",
    "AA3",
]

weather_parts = []

for chunk in pd.read_csv(WEATHER_PATH, usecols=weather_cols, chunksize=200_000, low_memory=False):
    chunk = chunk[chunk["STATION"].isin(NOAA_STATIONS.keys())].copy()
    if chunk.empty:
        continue

    chunk["weather_time_utc"] = pd.to_datetime(chunk["DATE"], utc=True, errors="coerce")
    chunk["pickup_hour"] = (
        chunk["weather_time_utc"]
        .dt.tz_convert("America/New_York")
        .dt.tz_localize(None)
        .dt.floor("h")
    )

    chunk = chunk[
        (chunk["pickup_hour"] >= "2023-01-01")
        & (chunk["pickup_hour"] < "2024-01-01")
    ].copy()

    chunk["temp_c"] = chunk["TMP"].apply(lambda x: parse_signed_tenths(x, 9999))
    chunk["dew_c"] = chunk["DEW"].apply(lambda x: parse_signed_tenths(x, 9999))
    chunk["pressure_hpa"] = chunk["SLP"].apply(lambda x: parse_signed_tenths(x, 99999))
    chunk["wind_speed_mps"] = chunk["WND"].apply(parse_wind_speed_mps)
    chunk["visibility_m"] = chunk["VIS"].apply(parse_visibility_m)

    precip_cols = []
    for col in ["AA1", "AA2", "AA3"]:
        parsed_col = f"{col.lower()}_precip_mm"
        chunk[parsed_col] = chunk[col].apply(parse_one_hour_precip_mm)
        precip_cols.append(parsed_col)

    chunk["precip_mm"] = chunk[precip_cols].apply(first_non_null_max, axis=1)

    weather_parts.append(
        chunk[
            [
                "STATION",
                "pickup_hour",
                "temp_c",
                "dew_c",
                "pressure_hpa",
                "wind_speed_mps",
                "visibility_m",
                "precip_mm",
            ]
        ]
    )

weather_observed = pd.concat(weather_parts, ignore_index=True)

weather_hourly = (
    weather_observed.groupby(["STATION", "pickup_hour"], as_index=False)
    .agg(
        temp_c=("temp_c", "mean"),
        dew_c=("dew_c", "mean"),
        pressure_hpa=("pressure_hpa", "mean"),
        wind_speed_mps=("wind_speed_mps", "mean"),
        visibility_m=("visibility_m", "mean"),
        precip_mm=("precip_mm", "max"),
    )
)

weather_hourly = weather_hourly.merge(station_metadata, on="STATION", how="left")
weather_hourly.head()

,STATION,pickup_hour,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm,weather_station_key,weather_station_name,usaf,wban,taxi_zone_note
0,72502014734,2023-01-01 00:00:00,11.700000,10.6,1007.2,3.1,4828.000000,0.0,newark,"Newark Liberty International Airport, NJ US",725020,14734,Used for Newark Airport taxi zone 1
1,72502014734,2023-01-01 01:00:00,11.133333,10.0,1007.2,3.1,6964.666667,0.0,newark,"Newark Liberty International Airport, NJ US",725020,14734,Used for Newark Airport taxi zone 1
2,72502014734,2023-01-01 02:00:00,9.400000,8.3,1008.1,2.6,6437.000000,0.0,newark,"Newark Liberty International Airport, NJ US",725020,14734,Used for Newark Airport taxi zone 1
3,72502014734,2023-01-01 03:00:00,8.900000,7.8,1008.4,2.1,8047.000000,0.0,newark,"Newark Liberty International Airport, NJ US",725020,14734,Used for Newark Airport taxi zone 1
4,72502014734,2023-01-01 04:00:00,9.400000,8.3,1008.9,3.6,8047.000000,0.0,newark,"Newark Liberty International Airport, NJ US",725020,14734,Used for Newark Airport taxi zone 1


### Fill small weather gaps

The model needs a weather value for every hour. This cell creates a complete station-hour grid and fills gaps by interpolation within each station, then forward/backward fill for any remaining edge gaps.

In [8]:
# Build a full station-hour grid so every taxi hour can receive a weather value.
station_hour_grid = pd.MultiIndex.from_product(
    [station_metadata["STATION"], all_hours],
    names=["STATION", "pickup_hour"],
).to_frame(index=False)

weather_hourly = station_hour_grid.merge(
    weather_hourly,
    on=["STATION", "pickup_hour"],
    how="left",
).merge(station_metadata, on="STATION", how="left", suffixes=("", "_station"))

for col in ["weather_station_key", "weather_station_name", "usaf", "wban", "taxi_zone_note"]:
    station_col = f"{col}_station"
    if station_col in weather_hourly.columns:
        weather_hourly[col] = weather_hourly[col].fillna(weather_hourly[station_col])
        weather_hourly = weather_hourly.drop(columns=[station_col])

numeric_weather_cols = [
    "temp_c",
    "dew_c",
    "pressure_hpa",
    "wind_speed_mps",
    "visibility_m",
    "precip_mm",
]

weather_hourly = weather_hourly.sort_values(["STATION", "pickup_hour"])
weather_hourly[numeric_weather_cols] = (
    weather_hourly.groupby("STATION")[numeric_weather_cols]
    .transform(lambda s: s.interpolate(limit_direction="both"))
)

weather_hourly["has_precipitation"] = (weather_hourly["precip_mm"] > 0).astype(int)

weather_missing_summary = weather_hourly.groupby("weather_station_key")[numeric_weather_cols].apply(
    lambda frame: frame.isna().mean().round(4)
)

weather_missing_summary

,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm
weather_station_key,,,,,,
central_park,0.0,0.0,0.0,0.0,0.0,0.0
jfk,0.0,0.0,0.0,0.0,0.0,0.0
laguardia,0.0,0.0,0.0,0.0,0.0,0.0
newark,0.0,0.0,0.0,0.0,0.0,0.0


## 6. Assign weather stations to taxi zones and join weather

Most zones receive Central Park weather because it is a long-running central NYC station. Airport zones receive airport-specific weather because airport taxi demand may be affected by localized airport conditions.

In [9]:
def assign_weather_station_key(pu_location_id):
    if pu_location_id == 132:
        return "jfk"
    if pu_location_id == 138:
        return "laguardia"
    if pu_location_id == 1:
        return "newark"
    return "central_park"


taxi_hourly["weather_station_key"] = taxi_hourly["PULocationID"].apply(assign_weather_station_key)

weather_features = weather_hourly[
    [
        "pickup_hour",
        "weather_station_key",
        "weather_station_name",
        "STATION",
        "usaf",
        "wban",
        "temp_c",
        "dew_c",
        "pressure_hpa",
        "wind_speed_mps",
        "visibility_m",
        "precip_mm",
        "has_precipitation",
    ]
]

taxi_hourly = taxi_hourly.merge(
    weather_features,
    on=["pickup_hour", "weather_station_key"],
    how="left",
)

taxi_hourly[["pickup_hour", "PULocationID", "Zone", "weather_station_name", "temp_c", "precip_mm"]].head()

,pickup_hour,PULocationID,Zone,weather_station_name,temp_c,precip_mm
0,2023-01-01,1,Newark Airport,"Newark Liberty International Airport, NJ US",11.700,0.0
1,2023-01-01,2,Jamaica Bay,"NY City Central Park, NY US",12.025,0.0
2,2023-01-01,3,Allerton/Pelham Gardens,"NY City Central Park, NY US",12.025,0.0
3,2023-01-01,4,Alphabet City,"NY City Central Park, NY US",12.025,0.0
4,2023-01-01,5,Arden Heights,"NY City Central Park, NY US",12.025,0.0


## 7. Prepare holiday features

Holidays are joined by date, not hour. The downloaded holiday file is useful, but I validate it because the raw file may contain date issues. For example, Memorial Day 2023 should be May 29, 2023.

For taxi demand, both actual holiday dates and observed federal leave dates can matter. For example, New Year's Day 2023 was January 1, while the observed federal leave date was January 2. This notebook flags either date as a holiday.

The final features are:

- `is_holiday`
- `holiday_name`
- `is_day_before_holiday`
- `is_day_after_holiday`

In [10]:
actual_holiday_col = "Date"
observed_holiday_col = "Pay and Leave Holiday Date (if SAT, then FRI if SUN  then MON)"

holidays_raw = pd.read_csv(HOLIDAY_PATH)
holidays = holidays_raw.copy()
holidays["actual_holiday_date"] = pd.to_datetime(holidays[actual_holiday_col], errors="coerce")
holidays["observed_holiday_date"] = pd.to_datetime(holidays[observed_holiday_col], errors="coerce")
holidays = holidays[holidays["actual_holiday_date"].dt.year == 2023].copy()

# Correct a known issue observed in the downloaded file
holidays.loc[holidays["Title"].eq("Memorial Day"), ["actual_holiday_date", "observed_holiday_date"]] = pd.Timestamp("2023-05-29")

actual_holidays = holidays[["Title", "actual_holiday_date"]].rename(
    columns={"Title": "holiday_name", "actual_holiday_date": "holiday_date"}
)
actual_holidays["holiday_date_type"] = "actual"

observed_holidays = holidays[["Title", "observed_holiday_date"]].rename(
    columns={"Title": "holiday_name", "observed_holiday_date": "holiday_date"}
)
observed_holidays["holiday_date_type"] = "observed"

holidays = pd.concat([actual_holidays, observed_holidays], ignore_index=True)
holidays = holidays.dropna(subset=["holiday_date"])
holidays = holidays[holidays["holiday_date"].dt.year == 2023]
holidays = (
    holidays.groupby("holiday_date", as_index=False)
    .agg(
        holiday_name=("holiday_name", lambda s: "; ".join(sorted(set(s)))),
        holiday_date_type=("holiday_date_type", lambda s: "; ".join(sorted(set(s)))),
    )
    .sort_values("holiday_date")
)

taxi_hourly["date"] = taxi_hourly["pickup_hour"].dt.normalize()

taxi_hourly = taxi_hourly.merge(
    holidays,
    left_on="date",
    right_on="holiday_date",
    how="left",
)

holiday_dates = set(holidays["holiday_date"])

taxi_hourly["is_holiday"] = taxi_hourly["holiday_date"].notna().astype(int)
taxi_hourly["is_day_before_holiday"] = (
    taxi_hourly["date"] + pd.Timedelta(days=1)
).isin(holiday_dates).astype(int)
taxi_hourly["is_day_after_holiday"] = (
    taxi_hourly["date"] - pd.Timedelta(days=1)
).isin(holiday_dates).astype(int)
taxi_hourly["holiday_name"] = taxi_hourly["holiday_name"].fillna("None")

holidays

,holiday_date,holiday_name,holiday_date_type
0,2023-01-01,New Year's Day,actual
1,2023-01-02,New Year's Day,observed
2,2023-01-16,"Birthday of Martin Luther King, Jr",actual; observed
3,2023-02-20,Washington's Birthday,actual; observed
4,2023-05-29,Memorial Day,actual; observed
5,2023-06-19,Juneteenth National Independence Day,actual; observed
6,2023-07-04,Independence Day,actual; observed
7,2023-09-04,Labor Day,actual; observed
8,2023-10-09,Columbus Day,actual; observed
9,2023-11-10,Veterans Day,observed


## 8. Add calendar features

Taxi demand has strong hourly, weekly, and seasonal patterns. Cyclical encodings represent repeating variables such as hour of day and day of week without creating artificial breaks between 23:00 and 00:00 or Sunday and Monday.

In [20]:
taxi_hourly["hour"] = taxi_hourly["pickup_hour"].dt.hour
taxi_hourly["dayofweek"] = taxi_hourly["pickup_hour"].dt.dayofweek
taxi_hourly["dayofyear"] = taxi_hourly["pickup_hour"].dt.dayofyear
taxi_hourly["weekofyear"] = taxi_hourly["pickup_hour"].dt.isocalendar().week.astype(int)
taxi_hourly["month"] = taxi_hourly["pickup_hour"].dt.month
taxi_hourly["is_weekend"] = taxi_hourly["dayofweek"].isin([5, 6]).astype(int)

taxi_hourly["hour_sin"] = np.sin(2 * np.pi * taxi_hourly["hour"] / 24)
taxi_hourly["hour_cos"] = np.cos(2 * np.pi * taxi_hourly["hour"] / 24)
taxi_hourly["dow_sin"] = np.sin(2 * np.pi * taxi_hourly["dayofweek"] / 7)
taxi_hourly["dow_cos"] = np.cos(2 * np.pi * taxi_hourly["dayofweek"] / 7)
taxi_hourly["month_sin"] = np.sin(2 * np.pi * taxi_hourly["month"] / 12)
taxi_hourly["month_cos"] = np.cos(2 * np.pi * taxi_hourly["month"] / 12)

taxi_hourly[
    [
        "pickup_hour",
        "PULocationID",
        "Zone",
        "trip_count",
        "hour",
        "dayofweek",
        "month",
        "is_weekend",
        "hour_sin",
        "hour_cos",
    ]
].head()

,pickup_hour,PULocationID,Zone,trip_count,hour,dayofweek,month,is_weekend,hour_sin,hour_cos
0,2023-01-01 00:00:00,1,Newark Airport,0.0,0,6,1,1,0.000000,1.000000
1,2023-01-01 01:00:00,1,Newark Airport,0.0,1,6,1,1,0.258819,0.965926
2,2023-01-01 02:00:00,1,Newark Airport,0.0,2,6,1,1,0.500000,0.866025
3,2023-01-01 03:00:00,1,Newark Airport,0.0,3,6,1,1,0.707107,0.707107
4,2023-01-01 04:00:00,1,Newark Airport,0.0,4,6,1,1,0.866025,0.500000


## 9. Add lag and rolling demand features

Lag features are historical demand values for the same zone. They are usually the strongest predictors in demand forecasting.

The rolling features use `shift(1)` before calculating the rolling mean. This prevents data leakage because the model cannot use the current hour's actual demand to predict the current hour.

In [12]:
taxi_hourly = taxi_hourly.sort_values(["PULocationID", "pickup_hour"]).reset_index(drop=True)

# Each lag uses past demand from the same pickup zone only.
for lag in [1, 2, 3, 24, 48, 72, 168]:
    taxi_hourly[f"trip_count_lag_{lag}"] = taxi_hourly.groupby("PULocationID")["trip_count"].shift(lag)

# Shift first so rolling means never include the current hour's true demand.
shifted_demand = taxi_hourly.groupby("PULocationID")["trip_count"].shift(1)

taxi_hourly["trip_count_rolling_24_mean"] = (
    shifted_demand.groupby(taxi_hourly["PULocationID"]).rolling(24, min_periods=6).mean().reset_index(level=0, drop=True)
)
taxi_hourly["trip_count_rolling_168_mean"] = (
    shifted_demand.groupby(taxi_hourly["PULocationID"]).rolling(168, min_periods=24).mean().reset_index(level=0, drop=True)
)

taxi_hourly[["pickup_hour", "PULocationID", "trip_count", "trip_count_lag_24", "trip_count_lag_168", "trip_count_rolling_24_mean"]].head(30)

,pickup_hour,PULocationID,trip_count,trip_count_lag_24,trip_count_lag_168,trip_count_rolling_24_mean
0,2023-01-01 00:00:00,1,0.0,NaN,NaN,NaN
1,2023-01-01 01:00:00,1,0.0,NaN,NaN,NaN
2,2023-01-01 02:00:00,1,0.0,NaN,NaN,NaN
3,2023-01-01 03:00:00,1,0.0,NaN,NaN,NaN
4,2023-01-01 04:00:00,1,0.0,NaN,NaN,NaN
5,2023-01-01 05:00:00,1,1.0,NaN,NaN,NaN
6,2023-01-01 06:00:00,1,0.0,NaN,NaN,0.166667
7,2023-01-01 07:00:00,1,0.0,NaN,NaN,0.142857
8,2023-01-01 08:00:00,1,1.0,NaN,NaN,0.125000
9,2023-01-01 09:00:00,1,0.0,NaN,NaN,0.222222


## 10. Final quality checks and save prepared data

The prepared dataset is saved as parquet for efficient loading in the next stages: model selection, training, and evaluation.

In [13]:
prepared_path = PROCESSED_DIR / "hourly_taxi_demand_2023_prepared.parquet"
cleaning_summary_path = PROCESSED_DIR / "taxi_cleaning_summary.csv"
weather_station_path = PROCESSED_DIR / "weather_station_metadata.csv"
holiday_path = PROCESSED_DIR / "holidays_2023_cleaned.csv"

quality_report = {
    "prepared_rows": len(taxi_hourly),
    "prepared_columns": taxi_hourly.shape[1],
    "min_pickup_hour": taxi_hourly["pickup_hour"].min(),
    "max_pickup_hour": taxi_hourly["pickup_hour"].max(),
    "zones": taxi_hourly["PULocationID"].nunique(),
    "missing_weather_rows": int(taxi_hourly[numeric_weather_cols].isna().any(axis=1).sum()),
    "total_trip_count": int(taxi_hourly["trip_count"].sum()),
}

taxi_hourly.to_parquet(prepared_path, index=False)
cleaning_summary.to_csv(cleaning_summary_path, index=False)
station_metadata.to_csv(weather_station_path, index=False)
holidays.to_csv(holiday_path, index=False)

quality_report

{'prepared_rows': 2303880,
 'prepared_columns': 51,
 'min_pickup_hour': Timestamp('2023-01-01 00:00:00'),
 'max_pickup_hour': Timestamp('2023-12-31 23:00:00'),
 'zones': 263,
 'missing_weather_rows': 0,
 'total_trip_count': 37267702}

## 11. output

The main output is:

`data/processed/hourly_taxi_demand_2023_prepared.parquet`

This file is to be used during model selection.